In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from torchvision.utils import make_grid

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline


In [3]:
#convert image file into tensor at 4 dimenssion
## Hight,width,color channels
transforms=transforms.ToTensor()

In [4]:
## Train data
train_data=datasets.MNIST(root='/cnn_data',train=True,download=True,transform=transforms)

In [5]:
## test data
test_data=datasets.MNIST(root='/cnn_data',train=False,download=True,transform=transforms)

In [6]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: /cnn_data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [7]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: /cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [8]:
## create a small batch size for images

train_loader=DataLoader(train_data,batch_size=10,shuffle=True)
test_loader=DataLoader(test_data,batch_size=10,shuffle=False)

In [9]:
##define CNN
conv1=nn.Conv2d(1,6,3,1)
conv2=nn.Conv2d(6,16,3,1)

In [10]:
## grab 1 MINST record

for i,(X_train,y_train) in enumerate(train_data):
    break

In [11]:
X_train.shape

torch.Size([1, 28, 28])

In [12]:
x=X_train.view(1,1,28,28)

In [13]:
## perform our first convolution
x=F.relu(conv1(x))# Rectifirfd linear unit for our activation fnction

In [14]:
x

tensor([[[[0.0934, 0.0934, 0.0934,  ..., 0.0934, 0.0934, 0.0934],
          [0.0934, 0.0934, 0.0934,  ..., 0.0934, 0.0934, 0.0934],
          [0.0934, 0.0934, 0.0934,  ..., 0.0934, 0.0934, 0.0934],
          ...,
          [0.0934, 0.0934, 0.0000,  ..., 0.0934, 0.0934, 0.0934],
          [0.0934, 0.0934, 0.0969,  ..., 0.0934, 0.0934, 0.0934],
          [0.0934, 0.0934, 0.0934,  ..., 0.0934, 0.0934, 0.0934]],

         [[0.1176, 0.1176, 0.1176,  ..., 0.1176, 0.1176, 0.1176],
          [0.1176, 0.1176, 0.1176,  ..., 0.1176, 0.1176, 0.1176],
          [0.1176, 0.1176, 0.1176,  ..., 0.1176, 0.1176, 0.1176],
          ...,
          [0.1176, 0.1176, 0.0000,  ..., 0.1176, 0.1176, 0.1176],
          [0.1176, 0.1176, 0.0891,  ..., 0.1176, 0.1176, 0.1176],
          [0.1176, 0.1176, 0.1176,  ..., 0.1176, 0.1176, 0.1176]],

         [[0.0595, 0.0595, 0.0595,  ..., 0.0595, 0.0595, 0.0595],
          [0.0595, 0.0595, 0.0595,  ..., 0.0595, 0.0595, 0.0595],
          [0.0595, 0.0595, 0.0595,  ..., 0

In [16]:
x.shape

torch.Size([1, 6, 26, 26])

In [17]:
# oas through the pooling layer
x=F.max_pool2d(x,2,2)## kernal of 2 and stride of 2

In [18]:
x.shape  ## 26/2=13

torch.Size([1, 6, 13, 13])

In [19]:
##do our second convolution layer 
x=F.relu(conv2(x))

In [20]:
x.shape ## again pixel decreases

torch.Size([1, 16, 11, 11])

In [ ]:
x=F.max_pool2d(x,2,2)

In [ ]:
x.shape ## 11/2=5.5=> 5

torch.Size([1, 16, 5, 5])

In [23]:
## building model for cnn

## model class

class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,6,3,1)
        self.conv2=nn.Conv2d(6,16,3,1)
        ## fully connected layer

        self.fc1=nn.Linear(5*5*16,120)
        self.fc2=nn.Linear(120,84)
        self.fc3=nn.Linear(84,10)

    def forward(self,X):
        X=F.relu(self.conv1(X))
        x=F.max_pool2d(X,2,2)
        #second pass
        X=F.relu(self.conv2(X))
        x=F.max_pool2d(X,2,2)

        ## Re-view to flatte it out
        X=X.view(-1,16*5*5)


        ## fully connected layers
        X=F.relu(self.fc1(X))
        X=self.relu(self.fc2(X))
        X=self.fc3(X)
        return F.log_softmax(X,dim=1)



In [24]:
## create an intance of model

torch.manual_seed(41)
model=ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [25]:
#loss fuction optimizer
criterian=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)